# Train XGBoost Binary Screening Model

This notebook trains a binary XGBoost screening model for urgency triage:
- `0` = `routine-review`
- `1` = `immediate-attention`

The purpose is to create a conservative hospital screening classifier that catches as many urgent incidents as possible before detailed human review.


## Operational Framing

Collapsing the original 4-tier urgency scheme into two classes makes the task much easier for the model:
- `Low` and `Medium` become one routine bucket
- `High` and `Critical` become one urgent bucket

This removes the extreme sparsity problem around the `Low` class and turns the model into a screening filter rather than a fine-grained triage engine.

The main safety risk is no longer confusion between nearby urgency levels. The risk becomes missing the boundary case where an actually urgent incident is sent into the routine queue. For that reason, this notebook does not optimize for plain accuracy. It focuses on:
- recall for the `immediate-attention` class
- PR-AUC for threshold selection
- a lowered decision threshold instead of the default `0.50`


In [1]:
from pathlib import Path
import json
import pickle

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, average_precision_score, classification_report, confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "train":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "incidents_feature_engineered_encoded.csv"
ARTIFACTS_DIR = PROJECT_ROOT / "train" / "artifacts"
MODEL_ARTIFACT_DIR = ARTIFACTS_DIR / "xgboost_urgency_binary_screening"
MODEL_PATH = MODEL_ARTIFACT_DIR / "model.pkl"
METADATA_PATH = MODEL_ARTIFACT_DIR / "metadata.json"
THRESHOLD_REPORT_PATH = MODEL_ARTIFACT_DIR / "threshold_report.csv"
PREDICTIONS_PATH = MODEL_ARTIFACT_DIR / "test_predictions.csv"
RANDOM_SEED = 42
TARGET_RECALL = 0.95

MODEL_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
incidents_df = pd.read_csv(DATASET_PATH)

required_columns = {"feature_concatanation", "urgency", "urgency_label"}
missing_columns = required_columns - set(incidents_df.columns)
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

model_df = incidents_df[["feature_concatanation", "urgency", "urgency_label"]].dropna().copy()
model_df["feature_concatanation"] = model_df["feature_concatanation"].astype(str).str.strip()
model_df = model_df[model_df["feature_concatanation"] != ""]

positive_classes = {"High", "Critical"}
model_df["screening_label"] = model_df["urgency"].isin(positive_classes).astype(int)
model_df["screening_target"] = model_df["screening_label"].map({0: "routine-review", 1: "immediate-attention"})

print(f"Loaded {len(model_df)} training rows from {DATASET_PATH}")
model_df.head()


Loaded 3600 training rows from /home/lakshan/ssp/IMS/data/processed/incidents_feature_engineered_encoded.csv


,feature_concatanation,urgency,urgency_label,screening_label,screening_target
0,Hospital Administrator Consultant session reve...,Low,0,0,routine-review
1,Staff Nurse - Paediatric Ward Diet slip notes ...,High,2,1,immediate-attention
2,OPD Nursing Officer Large crowd surged at OPD ...,Medium,1,0,routine-review
3,ICU Charge Nurse ICU AHU-2 not holding 22C; te...,High,2,1,immediate-attention
4,"Nursing Officer-in-Charge, Surgical Ward Publi...",High,2,1,immediate-attention


In [3]:
X = model_df["feature_concatanation"]
y = model_df["screening_label"].astype(int)

label_mapping = {0: "routine-review", 1: "immediate-attention"}

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y,
)

X_validation, X_test, y_validation, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=y_temp,
)

pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "rows": [len(X_train), len(X_validation), len(X_test)],
        "positive_rows": [int(y_train.sum()), int(y_validation.sum()), int(y_test.sum())],
        "negative_rows": [int((1 - y_train).sum()), int((1 - y_validation).sum()), int((1 - y_test).sum())],
    }
)


,split,rows,positive_rows,negative_rows
0,train,2520,1655,865
1,validation,540,355,185
2,test,540,354,186


In [4]:
distribution_df = pd.DataFrame(
    {
        "count": model_df["screening_label"].value_counts().sort_index(),
    }
)
distribution_df.index = distribution_df.index.map(label_mapping)
distribution_df["ratio"] = distribution_df["count"] / distribution_df["count"].sum()
distribution_df


,count,ratio
screening_label,,
routine-review,1236,0.343333
immediate-attention,2364,0.656667


## Train Binary Screening Model

The model predicts the probability that an incident belongs to the urgent `immediate-attention` bucket. Threshold selection happens later.


In [5]:
screening_classifier = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_features=20000,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            XGBClassifier(
                objective="binary:logistic",
                n_estimators=350,
                max_depth=6,
                learning_rate=0.08,
                subsample=0.9,
                colsample_bytree=0.9,
                min_child_weight=2,
                reg_alpha=0.0,
                reg_lambda=1.0,
                eval_metric="logloss",
                random_state=RANDOM_SEED,
                tree_method="hist",
                n_jobs=-1,
            ),
        ),
    ]
)

screening_classifier.fit(X_train, y_train)
print("Finished training binary XGBoost screening classifier")


Finished training binary XGBoost screening classifier


## Threshold Selection

A hospital screening model should be conservative. Instead of using the default threshold of `0.50`, this notebook scans a threshold range and selects the threshold that:
- first satisfies the urgent-class recall target of at least `0.95`
- then maximizes precision
- then uses F1 as a tie-breaker

If no threshold reaches the recall target, the model falls back to the threshold with the highest recall and then highest precision.


In [6]:
validation_probabilities = screening_classifier.predict_proba(X_validation)[:, 1]
test_probabilities = screening_classifier.predict_proba(X_test)[:, 1]

threshold_candidates = np.round(np.arange(0.20, 0.61, 0.01), 2)
threshold_rows = []

for threshold in threshold_candidates:
    validation_predictions = (validation_probabilities >= threshold).astype(int)
    threshold_rows.append(
        {
            "threshold": float(threshold),
            "validation_precision": float(precision_score(y_validation, validation_predictions, zero_division=0)),
            "validation_recall": float(recall_score(y_validation, validation_predictions, zero_division=0)),
            "validation_f1": float(f1_score(y_validation, validation_predictions, zero_division=0)),
            "validation_accuracy": float(accuracy_score(y_validation, validation_predictions)),
        }
    )

threshold_report_df = pd.DataFrame(threshold_rows)

eligible_thresholds = threshold_report_df[threshold_report_df["validation_recall"] >= TARGET_RECALL]
if not eligible_thresholds.empty:
    selected_threshold_row = eligible_thresholds.sort_values(
        by=["validation_precision", "validation_f1", "threshold"],
        ascending=[False, False, True],
    ).iloc[0]
else:
    selected_threshold_row = threshold_report_df.sort_values(
        by=["validation_recall", "validation_precision", "validation_f1", "threshold"],
        ascending=[False, False, False, True],
    ).iloc[0]

selected_threshold = float(selected_threshold_row["threshold"])
validation_pr_auc = average_precision_score(y_validation, validation_probabilities)
test_pr_auc = average_precision_score(y_test, test_probabilities)

print(f"Selected threshold: {selected_threshold:.2f}")
print(f"Validation PR-AUC: {validation_pr_auc:.4f}")
print(f"Test PR-AUC: {test_pr_auc:.4f}")

threshold_report_df


Selected threshold: 0.31
Validation PR-AUC: 0.9061
Test PR-AUC: 0.9082


,threshold,validation_precision,validation_recall,validation_f1,validation_accuracy
0,0.20,0.694000,0.977465,0.811696,0.701852
1,0.21,0.698189,0.977465,0.814554,0.707407
2,0.22,0.700405,0.974648,0.815077,0.709259
3,0.23,0.701826,0.974648,0.816038,0.711111
4,0.24,0.703476,0.969014,0.815166,0.711111
5,0.25,0.713693,0.969014,0.821983,0.724074
6,0.26,0.718163,0.969014,0.824940,0.729630
7,0.27,0.720588,0.966197,0.825511,0.731481
8,0.28,0.726695,0.966197,0.829504,0.738889
9,0.29,0.727660,0.963380,0.829091,0.738889


In [7]:
default_validation_predictions = (validation_probabilities >= 0.50).astype(int)
selected_validation_predictions = (validation_probabilities >= selected_threshold).astype(int)
selected_test_predictions = (test_probabilities >= selected_threshold).astype(int)

default_validation_recall = recall_score(y_validation, default_validation_predictions, zero_division=0)
selected_validation_recall = recall_score(y_validation, selected_validation_predictions, zero_division=0)

test_accuracy = accuracy_score(y_test, selected_test_predictions)
test_precision = precision_score(y_test, selected_test_predictions, zero_division=0)
test_recall = recall_score(y_test, selected_test_predictions, zero_division=0)
test_f1 = f1_score(y_test, selected_test_predictions, zero_division=0)

test_report_dict = classification_report(
    y_test,
    selected_test_predictions,
    labels=[0, 1],
    target_names=[label_mapping[0], label_mapping[1]],
    zero_division=0,
    output_dict=True,
)

print(f"Default threshold recall (0.50): {default_validation_recall:.4f}")
print(f"Selected threshold recall ({selected_threshold:.2f}): {selected_validation_recall:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")
print(f"Test precision for immediate-attention: {test_precision:.4f}")
print(f"Test recall for immediate-attention: {test_recall:.4f}")
print(f"Test F1 for immediate-attention: {test_f1:.4f}")
print(f"Test PR-AUC: {test_pr_auc:.4f}")
print("\nTest classification report:\n")
print(
    classification_report(
        y_test,
        selected_test_predictions,
        labels=[0, 1],
        target_names=[label_mapping[0], label_mapping[1]],
        zero_division=0,
    )
)

test_confusion_matrix_df = pd.DataFrame(
    confusion_matrix(y_test, selected_test_predictions, labels=[0, 1]),
    index=[f"actual_{label_mapping[0]}", f"actual_{label_mapping[1]}"],
    columns=[f"pred_{label_mapping[0]}", f"pred_{label_mapping[1]}"],
)
test_confusion_matrix_df


Default threshold recall (0.50): 0.8648
Selected threshold recall (0.31): 0.9521
Test accuracy: 0.7296
Test precision for immediate-attention: 0.7222
Test recall for immediate-attention: 0.9548
Test F1 for immediate-attention: 0.8224
Test PR-AUC: 0.9082

Test classification report:

                     precision    recall  f1-score   support

     routine-review       0.78      0.30      0.43       186
immediate-attention       0.72      0.95      0.82       354

           accuracy                           0.73       540
          macro avg       0.75      0.63      0.63       540
       weighted avg       0.74      0.73      0.69       540



,pred_routine-review,pred_immediate-attention
actual_routine-review,56,130
actual_immediate-attention,16,338


## Workflow Fit

This binary model fits best when the urgent queue still receives human review. In that setup, the model acts as an automated screening layer:
- routine-looking incidents stay in the `routine-review` bucket
- anything with meaningful urgency probability is escalated into the `immediate-attention` bucket

That keeps the final fine-grained triage decision with staff while reducing the chance that a truly urgent incident waits in a routine queue.


In [8]:
with MODEL_PATH.open("wb") as model_file:
    pickle.dump(screening_classifier, model_file)

predictions_df = pd.DataFrame(
    {
        "text": X_test.reset_index(drop=True),
        "actual_label": y_test.reset_index(drop=True),
        "predicted_probability_immediate_attention": pd.Series(test_probabilities),
        "predicted_label": pd.Series(selected_test_predictions),
    }
)
predictions_df["actual_target"] = predictions_df["actual_label"].map(label_mapping)
predictions_df["predicted_target"] = predictions_df["predicted_label"].map(label_mapping)

metadata = {
    "dataset_path": str(DATASET_PATH),
    "model_artifact_dir": str(MODEL_ARTIFACT_DIR),
    "model_path": str(MODEL_PATH),
    "threshold_report_path": str(THRESHOLD_REPORT_PATH),
    "predictions_path": str(PREDICTIONS_PATH),
    "random_seed": RANDOM_SEED,
    "target_recall": TARGET_RECALL,
    "selected_threshold": selected_threshold,
    "positive_class": label_mapping[1],
    "negative_class": label_mapping[0],
    "train_rows": int(len(X_train)),
    "validation_rows": int(len(X_validation)),
    "test_rows": int(len(X_test)),
    "validation_pr_auc": float(validation_pr_auc),
    "test_pr_auc": float(test_pr_auc),
    "test_accuracy": float(test_accuracy),
    "test_precision_immediate_attention": float(test_precision),
    "test_recall_immediate_attention": float(test_recall),
    "test_f1_immediate_attention": float(test_f1),
    "feature_column": "feature_concatanation",
    "target_column": "screening_label",
    "target_name_column": "screening_target",
    "classification_report": test_report_dict,
    "confusion_matrix": test_confusion_matrix_df.to_dict(),
}

METADATA_PATH.write_text(json.dumps(metadata, indent=2))
threshold_report_df.to_csv(THRESHOLD_REPORT_PATH, index=False)
predictions_df.to_csv(PREDICTIONS_PATH, index=False)

print(f"Saved trained model to {MODEL_PATH}")
print(f"Saved metadata to {METADATA_PATH}")
print(f"Saved threshold report to {THRESHOLD_REPORT_PATH}")
print(f"Saved test predictions to {PREDICTIONS_PATH}")


Saved trained model to /home/lakshan/ssp/IMS/train/artifacts/xgboost_urgency_binary_screening/model.pkl
Saved metadata to /home/lakshan/ssp/IMS/train/artifacts/xgboost_urgency_binary_screening/metadata.json
Saved threshold report to /home/lakshan/ssp/IMS/train/artifacts/xgboost_urgency_binary_screening/threshold_report.csv
Saved test predictions to /home/lakshan/ssp/IMS/train/artifacts/xgboost_urgency_binary_screening/test_predictions.csv
